In [1]:
import os, torch

root_path = os.path.dirname(os.getcwd())
os.chdir(root_path)
from modules.utils.general import get_device

### Fetch Data

In [ ]:
from modules.data_pipeline.retrieval import BaseDatasetRetriever
retriever = BaseDatasetRetriever(dataset_name="svo-probes", data_root=root_path+'/data')

### Convert to Diagrams

In [ ]:
from modules.symbolic import text_processor
functor = text_processor.TextProcessor('/Users/tls/Desktop/Work/COMP0267/assignment_5/COMP0267_CW/bobcat')
functor.text2diagram(path=root_path+'/data/svo-probes/processed', dataset=retriever.data, text_labels=retriever.text_labels)

### Compile to Quantum Circuits

In [2]:
from modules.utils.general import load_pkl, store_pkl
from modules.utils.general import get_device
from modules.compilation.classical.neural import MLPCompiler

ansatz = MLPCompiler()
compile_id = "MPL_Tree"

In [3]:
path = os.path.join(root_path, f'data/svo-probes/processed/compiled/{compile_id}')
if not os.path.exists(path): os.mkdir(path)

splits = ['train', 'val', 'test', 'swap']
for split in splits:
    df = load_pkl(f'data/svo-probes/processed/{split}.pkl')
    compiled_df = ansatz.compile_dataset(df)
    compiled_df['corrected_sentence_symbols'] = [[] for _ in range(len(compiled_df))]
    store_pkl(compiled_df, os.path.join(path, f'{split}.pkl'))
    print(f"Compiled {split} dataset saved to {os.path.join(path, f'{split}.pkl')}")

Compiled train dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/MPL_Tree/train.pkl
Compiled val dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/MPL_Tree/val.pkl
Compiled test dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/MPL_Tree/test.pkl
Compiled swap dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/MPL_Tree/swap.pkl


### Load Towers

In [4]:
train_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/train.pkl')
val_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/val.pkl')
test_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/test.pkl')
swap_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/swap.pkl')

In [5]:
from modules.models.text.einsum_quantum import QCModel
from modules.models.vision.image_model import TTNImageModel
from modules.models.text.neural_model import MLPTextModel

BATCH_SIZE = 256
DEV = 'cpu' # get_device()

image_model = TTNImageModel(embedding_dim=512)
image_model.to(DEV)
image_model.compile_batch(batch_size=BATCH_SIZE)

text_model = MLPTextModel(out_dim=512)
for df in [train_df, val_df, test_df, swap_df]:
    text_model.from_df(df)
text_model.to(DEV)

MLPTextModel(
  (leaves): ParameterDict(
      (baby_n): Parameter containing: [torch.FloatTensor of size 512]
      (father_n): Parameter containing: [torch.FloatTensor of size 512]
      (meadow_n): Parameter containing: [torch.FloatTensor of size 512]
      (woman_n): Parameter containing: [torch.FloatTensor of size 512]
      (grass_n): Parameter containing: [torch.FloatTensor of size 512]
      (puppy_n): Parameter containing: [torch.FloatTensor of size 512]
      (water_n): Parameter containing: [torch.FloatTensor of size 512]
      (boy_n): Parameter containing: [torch.FloatTensor of size 512]
      (pier_n): Parameter containing: [torch.FloatTensor of size 512]
      (man_n): Parameter containing: [torch.FloatTensor of size 512]
      (child_n): Parameter containing: [torch.FloatTensor of size 512]
      (mother_n): Parameter containing: [torch.FloatTensor of size 512]
      (girl_n): Parameter containing: [torch.FloatTensor of size 512]
      (street_n): Parameter containing: 

In [6]:
from modules.data_pipeline.datasets import SVODataset, svo_collate_fn
from torch.utils.data import DataLoader
from torchvision.transforms import v2
# , num_workers=4, pin_memory=True
IMG_TRANSFORM = v2.Compose([v2.ToImage(),               
                            v2.ToDtype(torch.float32, scale=True),
                            v2.Resize((64, 64))])

train_dataset = SVODataset(train_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, shuffle=True)

val_dataset = SVODataset(val_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

test_dataset = SVODataset(test_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

swap_dataset = SVODataset(swap_df, 'data/svo-probes/raw/images.zip', image_transform=IMG_TRANSFORM)
swap_loader = DataLoader(swap_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn)

### Learn and Evaluate

In [7]:
from modules.models.fusion.criteria import InfoNCE
from modules.models.fusion.engine import ContrastiveTrainer, MMEvaluator

LR = 1e-3
optimizer = torch.optim.Adam(list(text_model.parameters()) + list(image_model.parameters()), lr=LR)
loss_fn = InfoNCE()

trainer = ContrastiveTrainer(image_model, text_model, optimizer, loss_fn, DEV)
evaluator = MMEvaluator(image_model, text_model, DEV)

In [8]:
import mlflow, time
from modules.utils.general import set_seed
from modules.models.fusion.engine import svo_mapper
from pathlib import Path

EPOCHS = 25
SEED = int.from_bytes(os.urandom(4))
set_seed(SEED)

exp_name = 'svo-probes'

run_name = f"{int(time.time())}"
checkpoint_path = f"./checkpoints/{exp_name}/{run_name}.pt"
Path(os.path.dirname(checkpoint_path)).mkdir(parents=True, exist_ok=True)
mlf_path = os.path.join(root_path, f'mlf_dbs/{exp_name}.db')


mlflow.pytorch.autolog()
mlflow.set_tracking_uri(f"sqlite:///{mlf_path}")
mlflow.set_experiment(exp_name)

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "temperature_parameter": loss_fn.temperature,
        "device_target": str(DEV),
        "seed": SEED
        })
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        loss = trainer.train_epoch(train_loader, svo_mapper)
        end_time = time.time()
        
        svo_acc = evaluator.evaluate_image_choice(val_loader)
        
        mlflow.log_metric("train_loss", loss, step=epoch)
        mlflow.log_metric("svo_acc", svo_acc, step=epoch)
        print(f"Epoch {epoch} | Loss: {loss:.4f} | SVO: {svo_acc:.4f} | Time: {end_time - start_time:.2f}s")

        torch.save({
            "image": image_model.state_dict(),
            "text": text_model.state_dict(),
            "epoch": epoch,
            "train_loss": loss,
            "val_lacc": svo_acc if val_loader else None
            }, checkpoint_path)

Epoch 0 | Loss: 5.8915 | SVO: 0.4981 | Time: 134.50s
Epoch 1 | Loss: 5.5802 | SVO: 0.4915 | Time: 131.68s
Epoch 2 | Loss: 5.5427 | SVO: 0.5418 | Time: 141.90s
Epoch 3 | Loss: 5.5275 | SVO: 0.5232 | Time: 145.33s
Epoch 4 | Loss: 5.5110 | SVO: 0.5276 | Time: 139.50s
Epoch 5 | Loss: 5.4917 | SVO: 0.5183 | Time: 143.89s
Epoch 6 | Loss: 5.4242 | SVO: 0.5314 | Time: 159.64s
Epoch 7 | Loss: 5.3295 | SVO: 0.5828 | Time: 155.55s
Epoch 8 | Loss: 5.2274 | SVO: 0.5796 | Time: 152.21s
Epoch 9 | Loss: 5.1027 | SVO: 0.5856 | Time: 152.65s
Epoch 10 | Loss: 4.9496 | SVO: 0.6233 | Time: 152.56s
Epoch 11 | Loss: 4.7662 | SVO: 0.6299 | Time: 147.38s
Epoch 12 | Loss: 4.5632 | SVO: 0.6178 | Time: 117.08s
Epoch 13 | Loss: 4.3488 | SVO: 0.6484 | Time: 116.20s
Epoch 14 | Loss: 4.1660 | SVO: 0.6698 | Time: 117.60s
Epoch 15 | Loss: 3.8877 | SVO: 0.6747 | Time: 134.78s
Epoch 16 | Loss: 3.5993 | SVO: 0.6840 | Time: 123.81s
Epoch 17 | Loss: 3.3921 | SVO: 0.6916 | Time: 121.88s
Epoch 18 | Loss: 3.1892 | SVO: 0.6802 